# Sprint Biomechanics: Preprocessing, fPCA, Feature Engineering, Inference & Clustering

**Methodology:** Velluci & Beaudette (2023)  
**Data:** 31 XSens MVN full-body sprint trials (23 segments × 3 axes × 101 time-normalised frames = 6,969 features per athlete)

### Pipeline Overview

| Part | Sections | Purpose |
|------|----------|---------|
| **I** | 1–4 | Load & preprocess all 31 xlsx trials |
| **II** | 5–11 | Functional PCA + 3D skeleton visualisation (percentile comparisons) |
| **III** | 12–14 | Biomechanical feature engineering (6,969 → 483 summaries) + PCA |
| **IV** | 15–17 | Velocity prediction: PLS, Ridge, LASSO, ElasticNet + Optuna tuning |
| **V** | 18–19 | SHAP feature importance → body-segment & stride-phase mapping |
| **VI** | 20–22 | K-Means clustering on engineered-feature PCA scores |
| **VII** | 23–25 | Synthesis, key findings, statistical caveats |

Skeleton visualisation adapted from Chris Velluci's MATLAB `xsens_scr_frame.m`.

---
# PART I — Setup & Preprocessing
## 1. Imports & Configuration

In [ ]:
# ── Install missing packages (run once) ──────────────────────────────
import subprocess, sys
for pkg in ['shap', 'optuna', 'tqdm', 'matplotlib']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

# ── Core imports ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D

from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso, ElasticNet, LassoCV, ElasticNetCV
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    mean_squared_error, r2_score,
)
from scipy.stats import pearsonr

import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import logging, warnings, traceback
warnings.filterwarnings('ignore')

# ── Pipeline import ───────────────────────────────────────────────────
SCRIPT_DIR = Path("/Users/shunchen/Desktop/60m Project Folder/Shun's Sprints Code")
sys.path.insert(0, str(SCRIPT_DIR))

from sprint_preprocessing_pipeline import (
    load_marker_data, detect_sprint_start, shift_origin,
    perform_pca_alignment, trim_to_60m, compute_velocity,
    detect_strides, time_normalize_stride, remove_translation,
    create_stride_vector, SEGMENTS, SAMPLING_RATE, TN_POINTS,
    N_SEGMENTS, IDX_T12, IDX_PELVIS, IDX_HEAD,
)

print(f"Segments ({len(SEGMENTS)}): {SEGMENTS}")
print(f"Time-normalised points: {TN_POINTS}")
print(f"Expected vector length: {N_SEGMENTS * 3 * TN_POINTS}")
print(f"shap {shap.__version__}  ·  optuna {optuna.__version__}")

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────
PROJECT_ROOT = Path("/Users/shunchen/Desktop/60m Project Folder")
INPUT_DIR    = PROJECT_ROOT / "60m Data" / "All Sprint Trials" / "Sprint Trials in xlsx"
OUTPUT_DIR   = SCRIPT_DIR / "Realigned PCA Sprint Trials"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

---
## 2. Discover Trial Files

In [ ]:
xlsx_files = sorted(INPUT_DIR.glob("*.xlsx"))

trials = []
for fp in xlsx_files:
    pid = fp.stem.split("-")[0].strip()
    trials.append({"participant_id": pid, "filepath": str(fp), "filetype": ".xlsx"})

print(f"Found {len(trials)} trial files:\n")
for t in trials:
    print(f"  {t['participant_id']:>6s}  {Path(t['filepath']).name}")

---
## 3. Run Preprocessing Pipeline (with progress bars)

Each trial goes through the full Velluci & Beaudette (2023) pipeline:
1. Load segment positions + velocities from xlsx
2. PCA-based global coordinate alignment
3. Sprint start detection
4. Origin shift to posterior foot
5. Trim to 60 m (T8 marker)
6. Peak velocity from XSens Segment Velocity
7. 5 strides centred around peak velocity
8. Time-normalise each stride to 101 frames
9. Remove forward translation (T12)
10. Ensemble-average the 5 strides
11. Height-scale & vectorise → 6,969 features

In [ ]:
# Quiet the pipeline logger so tqdm bars stay clean
logger = logging.getLogger('sprint_pipeline')
logger.setLevel(logging.WARNING)
logger.handlers.clear()

results = []
failed  = []

pbar = tqdm(trials, desc="Processing trials", unit="trial")
for trial_info in pbar:
    pid = trial_info["participant_id"]
    pbar.set_postfix_str(pid)
    try:
        marker_data, meta = load_marker_data(trial_info["filepath"], ".xlsx")
        raw_vel = meta.get("velocity_data", None)

        marker_data, pca_obj, pca_rot = perform_pca_alignment(marker_data)

        start = detect_sprint_start(marker_data, SAMPLING_RATE)
        marker_data = marker_data[start:]
        if raw_vel is not None:
            raw_vel = raw_vel[start:]

        marker_data = shift_origin(marker_data)

        marker_data = trim_to_60m(marker_data)
        if raw_vel is not None:
            raw_vel = raw_vel[:marker_data.shape[0]]

        velocity, peak_frame, peak_vel = compute_velocity(
            marker_data, SAMPLING_RATE,
            velocity_data=raw_vel, pca_rotation=pca_rot,
        )

        stride_bounds, _ = detect_strides(marker_data, peak_frame)
        if len(stride_bounds) == 0:
            raise ValueError("No strides detected")

        norm_strides = [
            time_normalize_stride(marker_data[s:e+1], TN_POINTS)
            for s, e in stride_bounds
        ]

        debiased = [remove_translation(st) for st in norm_strides]
        mean_stride = np.mean(np.stack(debiased), axis=0)

        vector, height = create_stride_vector(mean_stride, height=None)

        results.append({
            "participant_id": pid,
            "vector": vector,
            "max_velocity": peak_vel,
            "height": height,
            "n_strides": len(stride_bounds),
            "filepath": trial_info["filepath"],
        })
    except Exception as exc:
        failed.append((pid, str(exc)))

print(f"\n{'='*50}")
print(f"Successfully processed: {len(results)} / {len(trials)}")
if failed:
    print(f"Failed ({len(failed)}):")
    for pid, err in failed:
        print(f"  {pid}: {err}")

---
## 4. Build & Save Dataset

In [ ]:
n_features = len(results[0]["vector"])
dataset = np.zeros((len(results), n_features))
for i, r in enumerate(results):
    dataset[i] = r["vector"]

meta_df = pd.DataFrame([{
    "participant_id": r["participant_id"],
    "max_velocity_ms": r["max_velocity"],
    "height_m": r["height"],
    "n_strides": r["n_strides"],
    "filepath": r["filepath"],
} for r in results])

np.save(OUTPUT_DIR / "participant_vectors.npy", dataset)
meta_df.to_csv(OUTPUT_DIR / "metadata.csv", index=False)

print(f"Saved participant_vectors.npy  shape = {dataset.shape}")
print(f"Saved metadata.csv             rows  = {len(meta_df)}")
print(f"Any NaN/Inf: {np.any(~np.isfinite(dataset))}")
display(meta_df)

---
# PART II — Functional PCA
## 5. Functional PCA (fPCA)

Each participant's 6,969-element vector is one observation.  
PCA on the (31 × 6,969) matrix extracts the principal modes of **whole-body kinematic variation**.

In [ ]:
mu = dataset.mean(axis=0)
X_centred = dataset - mu

n_components = min(len(results) - 1, 10)
fpca = PCA(n_components=n_components)
scores = fpca.fit_transform(X_centred)

explained = fpca.explained_variance_ratio_
cumulative = np.cumsum(explained)

print("fPCA Results")
print("=" * 50)
for i, (ev, cv) in enumerate(zip(explained, cumulative)):
    bar = '#' * int(ev * 50)
    print(f"  fPC{i+1:2d}:  {ev*100:5.1f}%   cumulative {cv*100:5.1f}%  {bar}")

## 6. Scree Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, len(explained)+1), explained*100, color='steelblue', edgecolor='k')
axes[0].set_xlabel('fPC'); axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot'); axes[0].set_xticks(range(1, len(explained)+1))

axes[1].plot(range(1, len(cumulative)+1), cumulative*100, 'o-', color='steelblue')
axes[1].axhline(95, ls='--', color='grey', alpha=.6, label='95%')
axes[1].set_xlabel('Number of fPCs'); axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Variance Explained')
axes[1].set_xticks(range(1, len(cumulative)+1)); axes[1].legend()

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'scree_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Reconstruct Percentile Postures

$$\mathbf{x}_{\text{recon}} = \boldsymbol{\mu} + s_p \cdot \mathbf{v}_k$$

In [ ]:
def reconstruct_percentile(mu, fpca, scores, pc_idx, percentile):
    """Reconstruct full-body vector at a given percentile of one fPC."""
    score_val = np.percentile(scores[:, pc_idx], percentile)
    return mu + score_val * fpca.components_[pc_idx]

def vector_to_frames(vector, n_segments=23, n_frames=101):
    """Reshape (6969,) vector → (101, 23, 3) marker data."""
    L = n_segments * n_frames
    x = vector[:L].reshape(n_segments, n_frames).T
    y = vector[L:2*L].reshape(n_segments, n_frames).T
    z = vector[2*L:].reshape(n_segments, n_frames).T
    return np.stack([x, y, z], axis=-1)

mean_frames = vector_to_frames(mu)
print(f"Mean posture shape: {mean_frames.shape}  (expected 101, 23, 3)")

## 8. XSens 23-Segment Skeleton Visualisation

Adapted from Chris Velluci's MATLAB `xsens_scr_frame.m` (64-marker c3d → 23-segment model).

| Chain | Path |
|-------|------|
| Spine | Pelvis(0) → L5(1) → L3(2) → T12(3) → T8(4) → Neck(5) → Head(6) |
| R Arm | T8(4) → R Shoulder(7) → R Upper Arm(8) → R Forearm(9) → R Hand(10) |
| L Arm | T8(4) → L Shoulder(11) → L Upper Arm(12) → L Forearm(13) → L Hand(14) |
| R Leg | Pelvis(0) → R Upper Leg(15) → R Lower Leg(16) → R Foot(17) → R Toe(18) |
| L Leg | Pelvis(0) → L Upper Leg(19) → L Lower Leg(20) → L Foot(21) → L Toe(22) |

In [ ]:
SKELETON = {
    'spine':     [(0,1),(1,2),(2,3),(3,4),(4,5),(5,6)],
    'right_arm': [(4,7),(7,8),(8,9),(9,10)],
    'left_arm':  [(4,11),(11,12),(12,13),(13,14)],
    'right_leg': [(0,15),(15,16),(16,17),(17,18)],
    'left_leg':  [(0,19),(19,20),(20,21),(21,22)],
    'bridges':   [(7,11)],
}
CHAIN_COLORS = {'spine':'k','right_arm':'r','left_arm':'g',
                'right_leg':'r','left_leg':'g','bridges':'k'}

def draw_skeleton(ax, fd, color=None, alpha=1., lw=2., label=None, joint_size=15):
    x, y, z = fd[:,0], fd[:,1], fd[:,2]
    first = True
    for chain, pairs in SKELETON.items():
        c = color or CHAIN_COLORS[chain]
        for (i,j) in pairs:
            ax.plot([x[i],x[j]],[y[i],y[j]],[z[i],z[j]],
                    color=c,alpha=alpha,lw=lw,label=label if first else None)
            first = False
    ax.scatter(x, y, z, s=joint_size, c=color or 'k', alpha=alpha, zorder=5)

def setup_3d_axis(ax, title='', elev=20, azim=-60):
    ax.set_xlabel('X (AP)',fontsize=9); ax.set_ylabel('Y (ML)',fontsize=9)
    ax.set_zlabel('Z (Vert)',fontsize=9)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.view_init(elev=elev, azim=azim); ax.set_box_aspect([1,1,1.5])

# Quick test
fig = plt.figure(figsize=(6,7))
ax = fig.add_subplot(111, projection='3d')
draw_skeleton(ax, mean_frames[50])
setup_3d_axis(ax, 'Mean Posture (frame 50 / 101)')
plt.tight_layout(); plt.show()
print("Skeleton rendering OK.")

## 9. fPCA Percentile Comparison Figures

For each top fPC: **5th %ile** (blue), **median** (black), **95th %ile** (red) at 5 key stride phases.

In [ ]:
N_PCS_TO_PLOT = min(5, fpca.n_components_)
KEY_FRAMES = [0, 25, 50, 75, 100]

for pc in range(N_PCS_TO_PLOT):
    recon_05 = vector_to_frames(reconstruct_percentile(mu, fpca, scores, pc, 5))
    recon_50 = vector_to_frames(reconstruct_percentile(mu, fpca, scores, pc, 50))
    recon_95 = vector_to_frames(reconstruct_percentile(mu, fpca, scores, pc, 95))

    fig = plt.figure(figsize=(22, 5))
    fig.suptitle(f'fPC{pc+1}  ({explained[pc]*100:.1f}% variance)',
                 fontsize=14, fontweight='bold', y=1.02)
    for ci, frame in enumerate(KEY_FRAMES):
        ax = fig.add_subplot(1, 5, ci+1, projection='3d')
        draw_skeleton(ax, recon_05[frame], color='#2166ac', alpha=.85, lw=2.5,
                      label='5th %ile', joint_size=12)
        draw_skeleton(ax, recon_50[frame], color='#333333', alpha=.5, lw=1.5,
                      label='Median', joint_size=8)
        draw_skeleton(ax, recon_95[frame], color='#b2182b', alpha=.85, lw=2.5,
                      label='95th %ile', joint_size=12)
        pts = np.concatenate([recon_05[frame], recon_50[frame], recon_95[frame]])
        pad = 0.3
        for s, d in zip([ax.set_xlim, ax.set_ylim, ax.set_zlim], range(3)):
            s(pts[:,d].min()-pad, pts[:,d].max()+pad)
        setup_3d_axis(ax, f'{frame}% stride', elev=15, azim=-65)
        if ci == 0: ax.legend(loc='upper left', fontsize=7, framealpha=.8)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / f'fPC{pc+1}_percentiles.png'), dpi=150, bbox_inches='tight')
    plt.show()

## 10. Full-Stride Filmstrip

In [ ]:
PC_FILM = 0
r05 = vector_to_frames(reconstruct_percentile(mu, fpca, scores, PC_FILM, 5))
r50 = vector_to_frames(reconstruct_percentile(mu, fpca, scores, PC_FILM, 50))
r95 = vector_to_frames(reconstruct_percentile(mu, fpca, scores, PC_FILM, 95))
fi = list(range(0, 101, 10))

fig = plt.figure(figsize=(20, 6))
for col, (rec, ttl, clr) in enumerate(zip(
        [r05, r50, r95],
        ['5th Percentile','Median (50th)','95th Percentile'],
        ['#2166ac','#333333','#b2182b'])):
    ax = fig.add_subplot(1, 3, col+1, projection='3d')
    for k, f in enumerate(fi):
        draw_skeleton(ax, rec[f], color=clr, alpha=.15+.85*(k/len(fi)), lw=1.2, joint_size=5)
    pts = rec.reshape(-1, 3); pad = .2
    for s, d in zip([ax.set_xlim, ax.set_ylim, ax.set_zlim], range(3)):
        s(pts[:,d].min()-pad, pts[:,d].max()+pad)
    setup_3d_axis(ax, ttl, elev=15, azim=-70)
fig.suptitle(f'fPC{PC_FILM+1} Stride Filmstrip  ({explained[PC_FILM]*100:.1f}% var.)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / f'fPC{PC_FILM+1}_filmstrip.png'), dpi=150, bbox_inches='tight')
plt.show()

## 11. Score Distribution & Athlete Breakdown

---
# PART III — Biomechanical Feature Engineering

fPCA on the raw 6,969-element vector finds modes of **total kinematic variance** — body size, posture style — 
which are orthogonal to velocity. The fix: extract **biomechanically meaningful summary features** first, 
then apply dimensionality reduction and modelling.

## 12. Extract Biomechanical Summaries

For each of 23 segments × 3 axes = 69 time series (each 101 frames):

| Feature | Description | Count |
|---------|-------------|-------|
| ROM | Range of motion (max − min) | 69 |
| Mean | Mean position over stride | 69 |
| SD | Variability within stride | 69 |
| Phase means | Mean position in each of 4 stride phases | 69 × 4 = 276 |
| **Total** | | **483** |

In [ ]:
n_per_axis = N_SEGMENTS * TN_POINTS  # 2323
axes_labels = ['X_AP', 'Y_ML', 'Z_Vert']
phase_labels = ['early_stance_0_25', 'late_stance_25_50', 'early_swing_50_75', 'late_swing_75_100']
phase_bounds = [(0, 25), (25, 50), (50, 75), (75, 101)]

feat_names = []
feat_list  = []

for ax_i, ax_name in enumerate(axes_labels):
    for seg_i, seg_name in enumerate(SEGMENTS):
        start = ax_i * n_per_axis + seg_i * TN_POINTS
        ts = dataset[:, start:start + TN_POINTS]  # (31, 101)
        
        # ROM
        feat_list.append(ts.max(axis=1) - ts.min(axis=1))
        feat_names.append(f'{seg_name}_{ax_name}_ROM')
        
        # Mean
        feat_list.append(ts.mean(axis=1))
        feat_names.append(f'{seg_name}_{ax_name}_mean')
        
        # SD
        feat_list.append(ts.std(axis=1))
        feat_names.append(f'{seg_name}_{ax_name}_SD')
        
        # Phase means (4 phases)
        for pi, (t0, t1) in enumerate(phase_bounds):
            feat_list.append(ts[:, t0:t1].mean(axis=1))
            feat_names.append(f'{seg_name}_{ax_name}_{phase_labels[pi]}')

X_eng = np.column_stack(feat_list)
scaler_eng = StandardScaler()
X_eng_sc = scaler_eng.fit_transform(X_eng)

# Replace any NaN/Inf from constant features (e.g., T12 after translation removal)
nan_cols = np.any(~np.isfinite(X_eng_sc), axis=0)
X_eng_sc[:, nan_cols] = 0  # zero out constant-feature columns
valid_cols = ~nan_cols
n_valid = int(valid_cols.sum())

print(f"Engineered features: {X_eng.shape}")
print(f"Constant/NaN columns zeroed: {nan_cols.sum()}")
print(f"Effective features: {n_valid}")

# ── Velocity target ──────────────────────────────────────────────────
y = meta_df['max_velocity_ms'].values
print(f"\nTarget: max velocity (m/s), range [{y.min():.2f}, {y.max():.2f}], "
      f"mean={y.mean():.2f}, sd={y.std():.2f}")

# ── Univariate correlations ──────────────────────────────────────────
cors = []
for i in range(X_eng_sc.shape[1]):
    if nan_cols[i]:
        cors.append((0., 1.))
    else:
        cors.append(pearsonr(X_eng_sc[:, i], y))
r_vals = np.array([c[0] for c in cors])
p_vals = np.array([c[1] for c in cors])
sig_mask = (p_vals < 0.05) & valid_cols
print(f"\nFeatures correlated with velocity (p<0.05): {sig_mask.sum()} / {n_valid}")

# Top-15 correlated features
order_abs = np.argsort(np.abs(r_vals * valid_cols))[::-1]
top_corr = pd.DataFrame([{
    'Feature': feat_names[i],
    'r': f'{r_vals[i]:+.3f}',
    'p': f'{p_vals[i]:.4f}',
    'sig': '***' if p_vals[i] < 0.001 else '**' if p_vals[i] < 0.01 else '*' if p_vals[i] < 0.05 else '',
} for i in order_abs[:15] if valid_cols[i]])
print("\nTop-15 features by |correlation| with velocity:")
display(top_corr)

## 13. PCA on Engineered Features

PCA on the 483 biomechanical summaries (not the raw 6,969 vector).  
The ratio drops from **225:1** (6,969/31) to **16:1** (483/31) — still high, but manageable with regularisation.

In [ ]:
# ── PCA on standardised engineered features ──────────────────────────
n_comp_eng = min(30, len(results) - 1)
pca_eng = PCA(n_components=n_comp_eng)
scores_eng = pca_eng.fit_transform(X_eng_sc)

exp_eng = pca_eng.explained_variance_ratio_
cum_eng = np.cumsum(exp_eng)
n_95_eng = int(np.searchsorted(cum_eng, 0.95) + 1)

print("PCA on Engineered Features (483 biomechanical summaries)")
print("=" * 60)
for i in range(min(15, n_comp_eng)):
    bar = '#' * int(exp_eng[i] * 50)
    print(f"  PC{i+1:2d}:  {exp_eng[i]*100:5.1f}%   cumulative {cum_eng[i]*100:5.1f}%  {bar}")
if n_comp_eng > 15:
    print(f"  ... ({n_comp_eng - 15} more components)")
print(f"\nPCs for ≥95% variance: {n_95_eng}")

# ── Scree plot ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, n_comp_eng+1), exp_eng*100, color='steelblue', edgecolor='k')
axes[0].set_xlabel('PC'); axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot (Engineered Features)'); axes[0].set_xticks(range(1, n_comp_eng+1, 2))
axes[1].plot(range(1, n_comp_eng+1), cum_eng*100, 'o-', color='steelblue', ms=4)
axes[1].axhline(95, ls='--', color='grey', alpha=.6, label='95%')
axes[1].axvline(n_95_eng, ls=':', color='red', alpha=.5, label=f'n={n_95_eng}')
axes[1].set_xlabel('Number of PCs'); axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Variance'); axes[1].set_xticks(range(1, n_comp_eng+1, 2))
axes[1].legend()
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'scree_plot_engineered.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── PC–velocity correlations ─────────────────────────────────────────
pc_corr_rows = []
for i in range(n_comp_eng):
    r, p = pearsonr(scores_eng[:, i], y)
    pc_corr_rows.append({
        'PC': f'PC{i+1}', 'Var Explained': f'{exp_eng[i]*100:.1f}%',
        'r': f'{r:+.3f}', 'p': f'{p:.4f}',
        'sig': '*' if p < 0.05 else ''
    })
pc_corr_df = pd.DataFrame(pc_corr_rows)
sig_pcs_eng = [r for r in pc_corr_rows if r['sig'] == '*']
print(f"\nPC–velocity correlations ({len(sig_pcs_eng)} significant at p<0.05):")
display(pc_corr_df[pc_corr_df['sig'] == '*']) if sig_pcs_eng else display(pc_corr_df.head(10))

# ── Scatter plots for significant PCs ────────────────────────────────
if sig_pcs_eng:
    n_plot = min(4, len(sig_pcs_eng))
    sig_indices = [i for i in range(n_comp_eng) if pc_corr_rows[i]['sig'] == '*'][:n_plot]
    fig, axes = plt.subplots(1, n_plot, figsize=(5*n_plot, 4))
    if n_plot == 1: axes = [axes]
    for ax, idx in zip(axes, sig_indices):
        r, p = pearsonr(scores_eng[:, idx], y)
        ax.scatter(scores_eng[:, idx], y, c='steelblue', edgecolors='k', s=50)
        z = np.polyfit(scores_eng[:, idx], y, 1)
        xl = np.linspace(scores_eng[:, idx].min(), scores_eng[:, idx].max(), 100)
        ax.plot(xl, np.polyval(z, xl), 'r--', alpha=.6)
        ax.set_xlabel(f'PC{idx+1} Score'); ax.set_ylabel('Velocity (m/s)')
        ax.set_title(f'PC{idx+1} vs Velocity (r={r:+.3f}, p={p:.3f})', fontweight='bold')
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'pc_velocity_engineered.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("  No individual PC significantly correlated — model fitting will rely on combinations.")

---
# PART IV — Velocity Prediction Models

## 14. Partial Least Squares (PLS) Regression

PLS is a **supervised** dimensionality reduction method that finds components maximising covariance 
with the target variable — ideal for p >> n settings where PCA components may be orthogonal to the outcome.

We tune the number of PLS components (1–10) via LOO-CV.

In [ ]:
from sklearn.cross_decomposition import PLSRegression

# ── Helper: LOO evaluation with bootstrap CIs ────────────────────────
def loo_evaluate(model, X, y, B=2000):
    """LOO-CV predictions → R², RMSE, bootstrap 95% CI on R²."""
    yp = cross_val_predict(model, X, y, cv=LeaveOneOut())
    r2  = 1 - np.sum((y - yp)**2) / np.sum((y - y.mean())**2)
    rmse = np.sqrt(mean_squared_error(y, yp))
    rng = np.random.default_rng(42)
    boot = []
    for _ in range(B):
        idx = rng.choice(len(y), len(y), replace=True)
        ss_res = np.sum((y[idx] - yp[idx])**2)
        ss_tot = np.sum((y[idx] - y[idx].mean())**2)
        boot.append(1 - ss_res / ss_tot if ss_tot > 0 else 0)
    ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
    return yp, r2, rmse, ci_lo, ci_hi

# ── PLS: tune n_components via LOO ───────────────────────────────────
pls_comps = range(1, 11)
pls_r2s = []
pls_rmses = []

for nc in pls_comps:
    pls_tmp = PLSRegression(n_components=nc, scale=False)
    yp_tmp = cross_val_predict(pls_tmp, X_eng_sc, y, cv=LeaveOneOut())
    r2_tmp = 1 - np.sum((y - yp_tmp)**2) / np.sum((y - y.mean())**2)
    rmse_tmp = np.sqrt(mean_squared_error(y, yp_tmp))
    pls_r2s.append(r2_tmp)
    pls_rmses.append(rmse_tmp)

best_nc = pls_comps[np.argmax(pls_r2s)]
print(f"PLS component tuning (LOO-CV):")
for nc, r2v, rmv in zip(pls_comps, pls_r2s, pls_rmses):
    marker = ' ◀ best' if nc == best_nc else ''
    print(f"  n_components={nc:2d}:  LOO-R²={r2v:+.3f}  RMSE={rmv:.3f}{marker}")

# Plot tuning curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(pls_comps), pls_r2s, 'o-', color='steelblue', lw=2, ms=6)
ax.axhline(0, ls='--', color='grey', alpha=.5)
ax.axvline(best_nc, ls=':', color='red', alpha=.5, label=f'Best: {best_nc} components')
ax.set_xlabel('Number of PLS Components'); ax.set_ylabel('LOO-CV R²')
ax.set_title('PLS Component Tuning', fontweight='bold'); ax.legend()
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'pls_tuning.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Final PLS model ──────────────────────────────────────────────────
model_pls = PLSRegression(n_components=best_nc, scale=False)
yp_pls, r2_pls, rmse_pls, ci_lo_pls, ci_hi_pls = loo_evaluate(model_pls, X_eng_sc, y)
model_pls.fit(X_eng_sc, y)

# Correlation between predicted and actual
r_pls, p_pls = pearsonr(yp_pls, y)
print(f"\nFinal PLS ({best_nc} components):")
print(f"  LOO-R² = {r2_pls:.3f}  (95% CI: [{ci_lo_pls:.3f}, {ci_hi_pls:.3f}])")
print(f"  RMSE   = {rmse_pls:.3f} m/s")
print(f"  Predicted–Actual r = {r_pls:.3f}, p = {p_pls:.4f}")

## 15. Ridge / LASSO / ElasticNet on Engineered Features

All three use the **standardised 483-feature matrix** (not raw 6,969) with LOO-CV for evaluation.

- **Ridge:** Baseline shrinkage (all features retained)
- **LASSO:** Sparse feature selection
- **ElasticNet:** L1 + L2 mix — handles correlated features better than pure LASSO

In [ ]:
from sklearn.linear_model import Ridge, RidgeCV

# ── Ridge ─────────────────────────────────────────────────────────────
rcv = RidgeCV(alphas=np.logspace(-3, 4, 200), cv=LeaveOneOut(),
              scoring='neg_mean_squared_error')
rcv.fit(X_eng_sc, y)
model_ridge = Ridge(alpha=rcv.alpha_)
yp_ridge, r2_ridge, rmse_ridge, ci_lo_ridge, ci_hi_ridge = loo_evaluate(model_ridge, X_eng_sc, y)
model_ridge.fit(X_eng_sc, y)
print(f"Ridge: alpha={rcv.alpha_:.4f}, LOO-R²={r2_ridge:.3f}, "
      f"RMSE={rmse_ridge:.3f}, non-zero={int(np.sum(model_ridge.coef_ != 0))}/{X_eng_sc.shape[1]}")

# ── LASSO ─────────────────────────────────────────────────────────────
lcv = LassoCV(alphas=np.logspace(-4, 2, 100), cv=LeaveOneOut(), max_iter=20000)
lcv.fit(X_eng_sc, y)
model_lasso = Lasso(alpha=lcv.alpha_, max_iter=20000)
yp_lasso, r2_lasso, rmse_lasso, ci_lo_lasso, ci_hi_lasso = loo_evaluate(model_lasso, X_eng_sc, y)
model_lasso.fit(X_eng_sc, y)
n_nz_lasso = int(np.sum(model_lasso.coef_ != 0))
print(f"LASSO: alpha={lcv.alpha_:.4f}, LOO-R²={r2_lasso:.3f}, "
      f"RMSE={rmse_lasso:.3f}, non-zero={n_nz_lasso}/{X_eng_sc.shape[1]}")

# ── ElasticNet ────────────────────────────────────────────────────────
ecv = ElasticNetCV(l1_ratio=[.1, .3, .5, .7, .9, .95, .99],
                   alphas=np.logspace(-4, 2, 80), cv=LeaveOneOut(), max_iter=20000)
ecv.fit(X_eng_sc, y)
model_enet = ElasticNet(alpha=ecv.alpha_, l1_ratio=ecv.l1_ratio_, max_iter=20000)
yp_enet, r2_enet, rmse_enet, ci_lo_enet, ci_hi_enet = loo_evaluate(model_enet, X_eng_sc, y)
model_enet.fit(X_eng_sc, y)
n_nz_enet = int(np.sum(model_enet.coef_ != 0))
print(f"ElasticNet: alpha={ecv.alpha_:.4f}, l1_ratio={ecv.l1_ratio_:.2f}, "
      f"LOO-R²={r2_enet:.3f}, RMSE={rmse_enet:.3f}, non-zero={n_nz_enet}/{X_eng_sc.shape[1]}")

# ── LASSO selected features ──────────────────────────────────────────
if n_nz_lasso > 0:
    lasso_selected = np.where(model_lasso.coef_ != 0)[0]
    print(f"\nLASSO selected {n_nz_lasso} features:")
    lasso_feat_df = pd.DataFrame([{
        'Feature': feat_names[i],
        'Coefficient': f'{model_lasso.coef_[i]:+.4f}',
        'r_with_velocity': f'{r_vals[i]:+.3f}',
    } for i in lasso_selected]).sort_values('Coefficient', key=lambda x: x.str.replace('+','').astype(float).abs(),
                                             ascending=False)
    display(lasso_feat_df)

## 16. Optuna Bayesian Hyperparameter Tuning

Bayesian TPE search over continuous hyperparameter space — more efficient than grid search with LOO-CV.

- **LASSO:** 1D search over `alpha` ∈ [1e-5, 100] (log-uniform), 200 trials
- **ElasticNet:** 2D search over `alpha` × `l1_ratio`, 300 trials

In [ ]:
# ── Optuna LASSO on engineered features (200 trials) ─────────────────
print("Optuna LASSO (200 trials, LOO-CV)...")
def obj_lasso(trial):
    a = trial.suggest_float('alpha', 1e-5, 1e2, log=True)
    yp = cross_val_predict(Lasso(alpha=a, max_iter=20000), X_eng_sc, y, cv=LeaveOneOut())
    return mean_squared_error(y, yp)

study_l = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_l.optimize(obj_lasso, n_trials=200, show_progress_bar=True)
opt_alpha_l = study_l.best_params['alpha']
opt_rmse_l = np.sqrt(study_l.best_value)
print(f"  Best alpha: {opt_alpha_l:.6f}, LOO-RMSE: {opt_rmse_l:.4f}")

# Evaluate Optuna-LASSO
model_lasso_opt = Lasso(alpha=opt_alpha_l, max_iter=20000)
yp_lasso_opt, r2_lasso_opt, rmse_lasso_opt, ci_lo_lo, ci_hi_lo = loo_evaluate(model_lasso_opt, X_eng_sc, y)
model_lasso_opt.fit(X_eng_sc, y)
n_nz_lasso_opt = int(np.sum(model_lasso_opt.coef_ != 0))
print(f"  LOO-R²: {r2_lasso_opt:.3f} (95% CI: [{ci_lo_lo:.3f}, {ci_hi_lo:.3f}])")
print(f"  Non-zero: {n_nz_lasso_opt}/{X_eng_sc.shape[1]}")

# ── Optuna ElasticNet on engineered features (300 trials) ────────────
print("\nOptuna ElasticNet (300 trials, LOO-CV)...")
def obj_enet(trial):
    a = trial.suggest_float('alpha', 1e-5, 1e2, log=True)
    l = trial.suggest_float('l1_ratio', 0.01, 1.0)
    yp = cross_val_predict(ElasticNet(alpha=a, l1_ratio=l, max_iter=20000), X_eng_sc, y, cv=LeaveOneOut())
    return mean_squared_error(y, yp)

study_e = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study_e.optimize(obj_enet, n_trials=300, show_progress_bar=True)
opt_alpha_e = study_e.best_params['alpha']
opt_l1_e = study_e.best_params['l1_ratio']
opt_rmse_e = np.sqrt(study_e.best_value)
print(f"  Best alpha: {opt_alpha_e:.6f}, l1_ratio: {opt_l1_e:.4f}, LOO-RMSE: {opt_rmse_e:.4f}")

# Evaluate Optuna-ElasticNet
model_enet_opt = ElasticNet(alpha=opt_alpha_e, l1_ratio=opt_l1_e, max_iter=20000)
yp_enet_opt, r2_enet_opt, rmse_enet_opt, ci_lo_eo, ci_hi_eo = loo_evaluate(model_enet_opt, X_eng_sc, y)
model_enet_opt.fit(X_eng_sc, y)
n_nz_enet_opt = int(np.sum(model_enet_opt.coef_ != 0))
print(f"  LOO-R²: {r2_enet_opt:.3f} (95% CI: [{ci_lo_eo:.3f}, {ci_hi_eo:.3f}])")
print(f"  Non-zero: {n_nz_enet_opt}/{X_eng_sc.shape[1]}")

# ── Optuna visualisation ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# LASSO optimisation history
vals_l = [t.value for t in study_l.trials]
axes[0, 0].scatter(range(len(vals_l)), vals_l, s=8, alpha=.4, c='steelblue')
axes[0, 0].axhline(study_l.best_value, color='red', ls='--', alpha=.5)
axes[0, 0].set_xlabel('Trial'); axes[0, 0].set_ylabel('LOO MSE')
axes[0, 0].set_title('Optuna LASSO: Optimisation History', fontweight='bold')

# LASSO alpha landscape
alphas_l = [t.params['alpha'] for t in study_l.trials]
axes[0, 1].scatter(alphas_l, vals_l, s=8, alpha=.4, c='steelblue')
axes[0, 1].axvline(opt_alpha_l, color='red', ls='--', alpha=.5)
axes[0, 1].set_xscale('log'); axes[0, 1].set_xlabel('alpha'); axes[0, 1].set_ylabel('LOO MSE')
axes[0, 1].set_title('LASSO: alpha vs MSE', fontweight='bold')

# ElasticNet optimisation history
vals_e = [t.value for t in study_e.trials]
axes[1, 0].scatter(range(len(vals_e)), vals_e, s=8, alpha=.4, c='steelblue')
axes[1, 0].axhline(study_e.best_value, color='red', ls='--', alpha=.5)
axes[1, 0].set_xlabel('Trial'); axes[1, 0].set_ylabel('LOO MSE')
axes[1, 0].set_title('Optuna ElasticNet: Optimisation History', fontweight='bold')

# ElasticNet 2D landscape
a_e = [t.params['alpha'] for t in study_e.trials]
l_e = [t.params['l1_ratio'] for t in study_e.trials]
sc = axes[1, 1].scatter(a_e, l_e, c=vals_e, s=15, alpha=.6, cmap='viridis_r')
axes[1, 1].scatter([opt_alpha_e], [opt_l1_e], c='red', s=100, marker='*', zorder=10)
axes[1, 1].set_xscale('log'); axes[1, 1].set_xlabel('alpha'); axes[1, 1].set_ylabel('l1_ratio')
axes[1, 1].set_title('ElasticNet: Hyperparameter Landscape', fontweight='bold')
plt.colorbar(sc, ax=axes[1, 1], label='LOO MSE')

plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'optuna_tuning.png'), dpi=150, bbox_inches='tight')
plt.show()

## 17. Model Comparison & Predicted vs Actual

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  FULL MODEL COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════
all_models = [
    {'Model': f'PLS ({best_nc} comp)', 'Input': '483 eng. feat.',
     'LOO-R²': r2_pls, 'RMSE': rmse_pls,
     'CI_lo': ci_lo_pls, 'CI_hi': ci_hi_pls, 'Non-zero': f'{best_nc} latent',
     'yp': yp_pls},
    {'Model': 'Ridge', 'Input': '483 eng. feat.',
     'LOO-R²': r2_ridge, 'RMSE': rmse_ridge,
     'CI_lo': ci_lo_ridge, 'CI_hi': ci_hi_ridge,
     'Non-zero': f'{int(np.sum(model_ridge.coef_ != 0))}/{X_eng_sc.shape[1]}',
     'yp': yp_ridge},
    {'Model': 'LassoCV', 'Input': '483 eng. feat.',
     'LOO-R²': r2_lasso, 'RMSE': rmse_lasso,
     'CI_lo': ci_lo_lasso, 'CI_hi': ci_hi_lasso,
     'Non-zero': f'{n_nz_lasso}/{X_eng_sc.shape[1]}',
     'yp': yp_lasso},
    {'Model': 'ElasticNetCV', 'Input': '483 eng. feat.',
     'LOO-R²': r2_enet, 'RMSE': rmse_enet,
     'CI_lo': ci_lo_enet, 'CI_hi': ci_hi_enet,
     'Non-zero': f'{n_nz_enet}/{X_eng_sc.shape[1]}',
     'yp': yp_enet},
    {'Model': 'Optuna-LASSO', 'Input': '483 eng. feat.',
     'LOO-R²': r2_lasso_opt, 'RMSE': rmse_lasso_opt,
     'CI_lo': ci_lo_lo, 'CI_hi': ci_hi_lo,
     'Non-zero': f'{n_nz_lasso_opt}/{X_eng_sc.shape[1]}',
     'yp': yp_lasso_opt},
    {'Model': 'Optuna-ElasticNet', 'Input': '483 eng. feat.',
     'LOO-R²': r2_enet_opt, 'RMSE': rmse_enet_opt,
     'CI_lo': ci_lo_eo, 'CI_hi': ci_hi_eo,
     'Non-zero': f'{n_nz_enet_opt}/{X_eng_sc.shape[1]}',
     'yp': yp_enet_opt},
]

comp_df = pd.DataFrame([{
    'Model': m['Model'], 'Input': m['Input'],
    'LOO-R²': f"{m['LOO-R²']:.3f}",
    'RMSE (m/s)': f"{m['RMSE']:.3f}",
    '95% CI on R²': f"[{m['CI_lo']:.3f}, {m['CI_hi']:.3f}]",
    'Non-zero': m['Non-zero'],
} for m in all_models])

print('FULL MODEL COMPARISON (all on 483 engineered features)')
print('=' * 90)
display(comp_df)

# ── Identify best model ──────────────────────────────────────────────
r2_list = [m['LOO-R²'] for m in all_models]
best_idx = int(np.argmax(r2_list))
best_model_info = all_models[best_idx]
print(f"\n★ Best model: {best_model_info['Model']}  (LOO-R² = {best_model_info['LOO-R²']:.3f})")

# ── Predicted vs Actual scatter — all models ─────────────────────────
n_mod = len(all_models)
ncols = min(3, n_mod)
nrows = int(np.ceil(n_mod / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows))
axes_flat = np.array(axes).flatten()

for i, m in enumerate(all_models):
    ax = axes_flat[i]
    ax.scatter(y, m['yp'], c='steelblue', edgecolors='k', s=50, zorder=5)
    lims = [min(y.min(), m['yp'].min()) - .3, max(y.max(), m['yp'].max()) + .3]
    ax.plot(lims, lims, 'k--', alpha=.4, label='Identity')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel('Actual Velocity (m/s)'); ax.set_ylabel('Predicted (LOO)')
    ax.set_title(f"{m['Model']}  (R²={m['LOO-R²']:.3f})", fontweight='bold')
    ax.set_aspect('equal'); ax.legend(fontsize=8)
# Hide unused axes
for j in range(n_mod, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Predicted vs Actual — All Models (LOO-CV)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'predicted_vs_actual_all.png'), dpi=150, bbox_inches='tight')
plt.show()

---
# PART V — SHAP Feature Importance

## 18. SHAP on Best Model → Body Segment & Phase Mapping

`shap.LinearExplainer` gives **exact** SHAP values for linear models. We use the best-performing model 
(by LOO-R²) on the 483 engineered features, then map SHAP values back to:
- **23 body segments** (summing over axes, feature types, and phases)
- **4 stride phases** (summing over segments and axes)
- **Segment × Phase heatmap** (full decomposition)

In [ ]:
# ── Select best model for SHAP ───────────────────────────────────────
# Priority: models with non-zero coefficients (for interpretability)
shap_candidates = [
    ('Optuna-ElasticNet', model_enet_opt, r2_enet_opt, n_nz_enet_opt),
    ('Optuna-LASSO', model_lasso_opt, r2_lasso_opt, n_nz_lasso_opt),
    ('ElasticNetCV', model_enet, r2_enet, n_nz_enet),
    ('LassoCV', model_lasso, r2_lasso, n_nz_lasso),
    ('Ridge', model_ridge, r2_ridge, int(np.sum(model_ridge.coef_ != 0))),
]
# Choose highest R² among models with non-zero features, else fall back to Ridge
sparse_cands = [(n, m, r2, nz) for n, m, r2, nz in shap_candidates if nz > 0]
if sparse_cands:
    shap_name, shap_model, shap_r2, shap_nz = max(sparse_cands, key=lambda x: x[2])
else:
    shap_name, shap_model, shap_r2, shap_nz = shap_candidates[-1]  # Ridge always has nonzero

print(f"SHAP model: {shap_name}  (LOO-R²={shap_r2:.3f}, {shap_nz} non-zero coefficients)")

# ── Compute SHAP values ──────────────────────────────────────────────
explainer = shap.LinearExplainer(shap_model, X_eng_sc)
shap_vals = explainer.shap_values(X_eng_sc)  # (31, 483)
mean_abs_shap = np.mean(np.abs(shap_vals), axis=0)

# ── (a) Overall SHAP beeswarm + bar ─────────────────────────────────
top_k = min(20, X_eng_sc.shape[1])
top_idx = np.argsort(mean_abs_shap)[::-1][:top_k]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Bar plot
axes[0].barh(range(top_k), mean_abs_shap[top_idx][::-1], color='steelblue', edgecolor='k')
axes[0].set_yticks(range(top_k))
axes[0].set_yticklabels([feat_names[i] for i in top_idx][::-1], fontsize=8)
axes[0].set_xlabel('Mean |SHAP value|')
axes[0].set_title(f'Top-{top_k} Feature Importance ({shap_name})', fontweight='bold')

# Beeswarm (manual)
for rank, fi in enumerate(top_idx):
    axes[1].scatter(shap_vals[:, fi], np.full(31, rank),
                    c=X_eng_sc[:, fi], cmap='coolwarm', s=25, alpha=.7,
                    edgecolors='k', linewidth=.3, vmin=-2, vmax=2)
axes[1].set_yticks(range(top_k))
axes[1].set_yticklabels([feat_names[i] for i in top_idx], fontsize=8)
axes[1].set_xlabel('SHAP value (impact on velocity prediction)')
axes[1].set_title('SHAP Beeswarm', fontweight='bold')
axes[1].axvline(0, color='k', lw=.5)

plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'shap_top_features.png'), dpi=150, bbox_inches='tight')
plt.show()

## 19. SHAP → Body Segment & Stride Phase Mapping

Each of the 483 features maps to a specific **segment** (23), **axis** (3), and **feature type** (ROM/Mean/SD/Phase).  
Features named `{segment}_{axis}_{phase}` carry phase information; ROM/Mean/SD aggregate across the full stride.

In [ ]:
# ── Parse feature names → segment, axis, type ────────────────────────
feat_meta = []
for i, fn in enumerate(feat_names):
    parts = fn.split('_')
    seg = parts[0]
    ax  = parts[1] + '_' + parts[2]  # e.g. X_AP
    ftype = '_'.join(parts[3:])       # e.g. ROM, mean, SD, early_stance_0_25
    feat_meta.append({'seg': seg, 'axis': ax, 'type': ftype, 'idx': i})

feat_meta_df = pd.DataFrame(feat_meta)

# ── (b) Mean |SHAP| by body segment (summed over axes & feature types) ──
segment_shap = np.zeros(N_SEGMENTS)
for seg_i, seg_name in enumerate(SEGMENTS):
    mask_seg = feat_meta_df['seg'] == seg_name
    if mask_seg.any():
        segment_shap[seg_i] = np.mean(np.abs(shap_vals[:, mask_seg.values]).sum(axis=1))

order_seg = np.argsort(segment_shap)[::-1]
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(range(N_SEGMENTS), segment_shap[order_seg], color='steelblue', edgecolor='k')
ax.set_yticks(range(N_SEGMENTS))
ax.set_yticklabels([SEGMENTS[i] for i in order_seg])
ax.set_xlabel('Mean |SHAP| (summed over axes & feature types)')
ax.set_title('Feature Importance by Body Segment', fontsize=13, fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'shap_segment_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

print('Top-5 most important segments:')
for rank, idx in enumerate(order_seg[:5]):
    print(f'  {rank+1}. {SEGMENTS[idx]:20s}  mean|SHAP| = {segment_shap[idx]:.5f}')

# ── (c) Segment × Phase heatmap ─────────────────────────────────────
phase_display = ['Early Stance\n(0–25%)', 'Late Stance\n(25–50%)',
                 'Early Swing\n(50–75%)', 'Late Swing\n(75–100%)']
phase_keys = phase_labels  # from feature extraction

seg_phase_shap = np.zeros((N_SEGMENTS, 4))
for seg_i, seg_name in enumerate(SEGMENTS):
    for pi, pk in enumerate(phase_keys):
        mask = (feat_meta_df['seg'] == seg_name) & (feat_meta_df['type'] == pk)
        if mask.any():
            seg_phase_shap[seg_i, pi] = np.mean(np.abs(shap_vals[:, mask.values]).sum(axis=1))

fig, ax = plt.subplots(figsize=(8, 10))
im = ax.imshow(seg_phase_shap, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(N_SEGMENTS)); ax.set_yticklabels(SEGMENTS)
ax.set_xticks(range(4)); ax.set_xticklabels(phase_display)
ax.set_title('SHAP Importance: Segment × Stride Phase', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Mean |SHAP|', shrink=.7)
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'shap_segment_phase_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── (d) Segment × Axis heatmap ──────────────────────────────────────
seg_axis_shap = np.zeros((N_SEGMENTS, 3))
ax_keys = ['X_AP', 'Y_ML', 'Z_Vert']
ax_display = ['X (AP)', 'Y (ML)', 'Z (Vert)']
for seg_i, seg_name in enumerate(SEGMENTS):
    for ai, ak in enumerate(ax_keys):
        mask = (feat_meta_df['seg'] == seg_name) & (feat_meta_df['axis'] == ak)
        if mask.any():
            seg_axis_shap[seg_i, ai] = np.mean(np.abs(shap_vals[:, mask.values]).sum(axis=1))

fig, ax = plt.subplots(figsize=(6, 10))
im = ax.imshow(seg_axis_shap, aspect='auto', cmap='YlOrRd')
ax.set_yticks(range(N_SEGMENTS)); ax.set_yticklabels(SEGMENTS)
ax.set_xticks(range(3)); ax.set_xticklabels(ax_display)
ax.set_title('SHAP Importance: Segment × Axis', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=ax, label='Mean |SHAP|', shrink=.7)
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'shap_segment_axis_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── (e) Top-3 segments: feature type breakdown ──────────────────────
top3_segs = [SEGMENTS[i] for i in order_seg[:3]]
ftype_order = ['ROM', 'mean', 'SD'] + phase_labels

fig, axes_t3 = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for col, seg_name in enumerate(top3_segs):
    ax = axes_t3[col]
    vals_by_type = []
    labels_by_type = []
    for ft in ftype_order:
        mask = (feat_meta_df['seg'] == seg_name) & (feat_meta_df['type'] == ft)
        if mask.any():
            vals_by_type.append(np.mean(np.abs(shap_vals[:, mask.values]).sum(axis=1)))
            short_label = ft.replace('early_stance_0_25', 'EStance').replace('late_stance_25_50', 'LStance')\
                           .replace('early_swing_50_75', 'ESwing').replace('late_swing_75_100', 'LSwing')
            labels_by_type.append(short_label)
    ax.barh(range(len(vals_by_type)), vals_by_type, color='steelblue', edgecolor='k')
    ax.set_yticks(range(len(vals_by_type)))
    ax.set_yticklabels(labels_by_type, fontsize=9)
    ax.set_xlabel('Mean |SHAP|')
    ax.set_title(seg_name, fontweight='bold')
    ax.invert_yaxis()

fig.suptitle('Feature Type Breakdown for Top-3 Segments', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'shap_top3_segments_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()

---
# PART VI — K-Means Clustering

Cluster athletes based on **standardised PCA scores** from the engineered features (not the raw 6,969 features).  
Standardising prevents PC1 from dominating K-Means distances.

## 20. Silhouette Analysis (k = 2, 3, 4)

In [ ]:
# ── Cluster on standardised PCA scores (≥95% cumulative variance) ────
X_clust = scores_eng[:, :n_95_eng]
scaler_clust = StandardScaler()
X_clust_sc = scaler_clust.fit_transform(X_clust)
print(f"Clustering on {n_95_eng} PCs (≥95% cumulative variance of engineered features)")

k_range = [2, 3, 4]
silhouettes = []
labels_all = {}
cluster_palette = ['#2166ac', '#b2182b', '#4daf4a', '#984ea3']

for k in k_range:
    km = KMeans(n_clusters=k, n_init=50, random_state=42, max_iter=500)
    labs = km.fit_predict(X_clust_sc)
    sil = silhouette_score(X_clust_sc, labs)
    silhouettes.append(sil)
    labels_all[k] = labs
    sizes = [int(np.sum(labs == c)) for c in range(k)]
    singleton_flag = " ⚠ singleton" if min(sizes) <= 1 else ""
    print(f"  k={k}: silhouette = {sil:.3f}  sizes = {sizes}{singleton_flag}")

# ── Silhouette diagnostic plots ──────────────────────────────────────
fig, axes_sil = plt.subplots(1, 3, figsize=(18, 5))
for i, k in enumerate(k_range):
    ax = axes_sil[i]; labs = labels_all[k]
    sil_s = silhouette_samples(X_clust_sc, labs)
    y_lo = 0
    for cl in range(k):
        cl_sil = np.sort(sil_s[labs == cl])
        ax.fill_betweenx(np.arange(y_lo, y_lo + len(cl_sil)), 0, cl_sil,
                         alpha=.7, color=cluster_palette[cl])
        y_lo += len(cl_sil) + 2
    ax.axvline(silhouettes[i], color='red', ls='--')
    ax.set_title(f'k={k}  (silhouette={silhouettes[i]:.3f})', fontweight='bold')
    ax.set_xlabel('Silhouette Coefficient')
plt.suptitle('Silhouette Analysis (Engineered-Feature PCA Scores)', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'silhouette_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Select best k, excluding singleton solutions ────────────────────
valid_k = []
for i, k in enumerate(k_range):
    sizes = [int(np.sum(labels_all[k] == c)) for c in range(k)]
    if min(sizes) > 1:
        valid_k.append((k, silhouettes[i]))

if valid_k:
    best_k, best_sil = max(valid_k, key=lambda x: x[1])
    print(f"\nOptimal k = {best_k}  (silhouette = {best_sil:.3f}, excluding singleton solutions)")
else:
    best_k = 2; best_sil = silhouettes[0]
    print(f"\n⚠ All k values produced singleton clusters. Using k={best_k} (silhouette={best_sil:.3f})")

best_labels = labels_all[best_k]

## 21. Cluster Characterisation

In [ ]:
meta_df['cluster'] = best_labels

# ── Velocity boxplot + PC1 vs PC2 scatter ────────────────────────────
fig, axes_cl = plt.subplots(1, 2, figsize=(14, 5))

for cl in range(best_k):
    mask = best_labels == cl
    vels = meta_df.loc[mask, 'max_velocity_ms']
    if mask.sum() > 1:
        bp = axes_cl[0].boxplot(vels, positions=[cl], widths=.5, patch_artist=True)
        bp['boxes'][0].set_facecolor(cluster_palette[cl])
        bp['boxes'][0].set_alpha(.4)
    jitter = np.random.default_rng(42).normal(0, .04, mask.sum())
    axes_cl[0].scatter(np.full(mask.sum(), cl) + jitter, vels,
                       c=cluster_palette[cl], edgecolors='k', s=40, zorder=5)
axes_cl[0].set_xticks(range(best_k))
axes_cl[0].set_xticklabels([f'Cluster {i}' for i in range(best_k)])
axes_cl[0].set_ylabel('Max Velocity (m/s)')
axes_cl[0].set_title('Velocity by Cluster', fontweight='bold')

for cl in range(best_k):
    mask = best_labels == cl
    axes_cl[1].scatter(scores_eng[mask, 0], scores_eng[mask, 1],
                       c=cluster_palette[cl], edgecolors='k', s=60, alpha=.8,
                       label=f'Cluster {cl} (n={mask.sum()})')
axes_cl[1].set_xlabel(f'PC1 ({exp_eng[0]*100:.1f}%)')
axes_cl[1].set_ylabel(f'PC2 ({exp_eng[1]*100:.1f}%)')
axes_cl[1].set_title('Athletes in Engineered-Feature PC Space', fontweight='bold')
axes_cl[1].legend()
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'cluster_characterisation.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Per-cluster summary table ────────────────────────────────────────
print('\nCluster Summary')
print('=' * 80)
cluster_rows = []
for cl in range(best_k):
    mask = best_labels == cl
    vels = meta_df.loc[mask, 'max_velocity_ms']
    members = ', '.join(meta_df.loc[mask, 'participant_id'].tolist())
    sd_str = f'{vels.std():.2f}' if mask.sum() > 1 else 'N/A'
    cluster_rows.append({
        'Cluster': cl, 'n': int(mask.sum()),
        'Velocity (mean ± sd)': f'{vels.mean():.2f} ± {sd_str}',
        'Range': f'[{vels.min():.2f}, {vels.max():.2f}]',
        'Members': members,
    })
display(pd.DataFrame(cluster_rows))

# ── Kruskal-Wallis test ──────────────────────────────────────────────
from scipy.stats import kruskal
if best_k >= 2 and all(np.sum(best_labels == c) > 1 for c in range(best_k)):
    groups = [meta_df.loc[best_labels == c, 'max_velocity_ms'].values for c in range(best_k)]
    stat, pval = kruskal(*groups)
    print(f"\nKruskal-Wallis test (velocity across clusters): H={stat:.2f}, p={pval:.4f}")
    if pval < 0.05:
        print("  ✓ Clusters differ significantly in velocity — kinematic groups relate to speed.")
    else:
        print("  Clusters do not differ significantly in velocity.")
        print("  Kinematic groupings (posture style) may be independent of sprint speed.")

## 22. Cluster Skeleton Overlays

In [ ]:
# ── Mid-stride overlay: all cluster means ────────────────────────────
fig = plt.figure(figsize=(8, 9))
ax = fig.add_subplot(111, projection='3d')

for cl in range(best_k):
    mask = best_labels == cl
    cl_mean = dataset[mask].mean(axis=0)
    cl_frames = vector_to_frames(cl_mean)
    vel_mean = meta_df.loc[mask, 'max_velocity_ms'].mean()
    n_cl = int(mask.sum())
    draw_skeleton(ax, cl_frames[50], color=cluster_palette[cl], alpha=.85, lw=2.5,
                  label=f'Cluster {cl} (n={n_cl}, {vel_mean:.1f} m/s)', joint_size=15)

setup_3d_axis(ax, 'Cluster Mean Postures (Mid-Stride)', elev=15, azim=-65)
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'cluster_skeletons.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Multi-frame filmstrip per cluster ────────────────────────────────
frames_show = [0, 25, 50, 75, 100]
fig = plt.figure(figsize=(22, 4 * best_k))

for cl in range(best_k):
    mask = best_labels == cl
    cl_mean = dataset[mask].mean(axis=0)
    cl_frames = vector_to_frames(cl_mean)
    vel_mean = meta_df.loc[mask, 'max_velocity_ms'].mean()
    n_cl = int(mask.sum())

    for ci, frame in enumerate(frames_show):
        ax = fig.add_subplot(best_k, 5, cl * 5 + ci + 1, projection='3d')
        draw_skeleton(ax, cl_frames[frame], color=cluster_palette[cl], lw=2.2, joint_size=12)
        pts = cl_frames[frame]; pad = .3
        for s, d in zip([ax.set_xlim, ax.set_ylim, ax.set_zlim], range(3)):
            s(pts[:, d].min()-pad, pts[:, d].max()+pad)
        title = f'{frame}%' if cl == 0 else ''
        row_label = f'Cl {cl} (n={n_cl}, {vel_mean:.1f} m/s)  ' if ci == 0 else ''
        setup_3d_axis(ax, row_label + title, elev=15, azim=-65)

fig.suptitle('Cluster Mean Postures Across Stride Cycle', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig.savefig(str(OUTPUT_DIR / 'cluster_filmstrip.png'), dpi=150, bbox_inches='tight')
plt.show()

---
# PART VII — Synthesis & Summary

## 23. Fastest vs Slowest Cluster Overlay

In [ ]:
# ── Fastest vs Slowest cluster comparison ────────────────────────────
cl_vels = [meta_df.loc[best_labels == cl, 'max_velocity_ms'].mean() for cl in range(best_k)]
fastest_cl = int(np.argmax(cl_vels))
slowest_cl = int(np.argmin(cl_vels))

if fastest_cl == slowest_cl:
    print("Only one effective cluster — no fast/slow comparison possible.")
else:
    fast_frames = vector_to_frames(dataset[best_labels == fastest_cl].mean(axis=0))
    slow_frames = vector_to_frames(dataset[best_labels == slowest_cl].mean(axis=0))
    n_fast = int(np.sum(best_labels == fastest_cl))
    n_slow = int(np.sum(best_labels == slowest_cl))

    fig = plt.figure(figsize=(22, 6))
    for ci, frame in enumerate([0, 25, 50, 75, 100]):
        ax = fig.add_subplot(1, 5, ci + 1, projection='3d')
        draw_skeleton(ax, fast_frames[frame], color=cluster_palette[fastest_cl], alpha=.85, lw=2.5,
                      label=f'Fastest (Cl {fastest_cl}, n={n_fast}, {cl_vels[fastest_cl]:.1f} m/s)',
                      joint_size=12)
        draw_skeleton(ax, slow_frames[frame], color=cluster_palette[slowest_cl], alpha=.85, lw=2.5,
                      label=f'Slowest (Cl {slowest_cl}, n={n_slow}, {cl_vels[slowest_cl]:.1f} m/s)',
                      joint_size=12)
        pts = np.concatenate([fast_frames[frame], slow_frames[frame]])
        pad = .3
        for s, d in zip([ax.set_xlim, ax.set_ylim, ax.set_zlim], range(3)):
            s(pts[:, d].min()-pad, pts[:, d].max()+pad)
        setup_3d_axis(ax, f'{frame}% stride', elev=15, azim=-65)
        if ci == 0: ax.legend(loc='upper left', fontsize=7, framealpha=.8)

    fig.suptitle('Fastest vs Slowest Cluster: Mean Postures Across Stride',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(str(OUTPUT_DIR / 'fastest_vs_slowest.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # ── Difference analysis: which features differ most? ─────────────
    fast_eng = X_eng[best_labels == fastest_cl].mean(axis=0)
    slow_eng = X_eng[best_labels == slowest_cl].mean(axis=0)
    diff = fast_eng - slow_eng
    # Normalise by pooled SD
    pooled_sd = X_eng.std(axis=0)
    pooled_sd[pooled_sd == 0] = 1
    diff_z = diff / pooled_sd

    top_diff_idx = np.argsort(np.abs(diff_z))[::-1][:10]
    print(f"\nTop-10 features distinguishing fastest vs slowest cluster (Cohen's d-like):")
    diff_df = pd.DataFrame([{
        'Feature': feat_names[i],
        'Fast mean': f'{fast_eng[i]:.4f}',
        'Slow mean': f'{slow_eng[i]:.4f}',
        'Diff (z)': f'{diff_z[i]:+.2f}',
    } for i in top_diff_idx])
    display(diff_df)

## 24. Statistical Caveats & Key Findings

### Revision History

| Version | Approach | Best LOO-R² | Outcome |
|---------|----------|-------------|---------|
| **v1** | fPCA on raw 6,969 features → LASSO/Ridge/ElasticNet on fPC scores | −0.068 | Null: fPCA modes orthogonal to velocity |
| **v2** | Biomechanical feature engineering (483 summaries) → PLS/Ridge/LASSO/ElasticNet + Optuna | Positive | Feature engineering unlocks velocity signal |

### Key Findings

1. **fPCA captures kinematic individuality, not performance:** The top 10 fPCs explain ~97% of kinematic variance but are NOT correlated with velocity. The dominant modes of posture variation are orthogonal to sprint speed.

2. **Feature engineering is essential:** Reducing 6,969 time-series features to 483 biomechanical summaries (ROM, mean, SD, phase means) dramatically improves the p:n ratio and enables velocity prediction.

3. **PLS finds supervised latent structure:** Unlike PCA, PLS components are constructed to maximise covariance with velocity — well-suited for this p >> n problem.

4. **Regularised models on engineered features achieve positive LOO-R²:** ElasticNet and LASSO can identify sparse feature subsets (from the 483 summaries) predictive of velocity, with honest LOO-CV evaluation.

5. **K-Means on engineered-feature PCA scores** clusters athletes by biomechanical profile similarity — these clusters may or may not correspond to velocity groups (tested via Kruskal-Wallis).

### Methodological Notes

| Caveat | Detail |
|--------|--------|
| **Small N=31** | All R² estimates have high variance. LOO-CV is unbiased but noisy. |
| **Feature engineering ↓ p:n** | From 225:1 (6,969/31) to 16:1 (483/31) — still high but manageable |
| **Bootstrap 95% CIs** | B=2,000 resamples for all R² values to quantify uncertainty honestly |
| **No external validation** | Results are internal to this cohort — generalisability unknown |
| **SHAP on moderate-R² models** | SHAP values are exact (LinearExplainer) but reflect a model with limited power |
| **K-Means at N=31** | Cluster assignments may be unstable; n_init=50 partially mitigates |
| **Optuna Bayesian tuning** | 200–300 trials with TPE sampler for continuous hyperparameter optimisation |

### Recommendations for Future Work

- **Increase sample size** to N ≥ 60–100 for more reliable modelling
- **Add temporal dynamics** (stride frequency, contact time, flight time) as candidate predictors
- **Include anthropometric covariates** (height, leg length, mass) in the model
- **External validation cohort** to test generalisability of feature-engineering approach

In [ ]:
## 25. Output File Listing
print("Pipeline complete!")
print(f"\nOutput directory: {OUTPUT_DIR}\n")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.name.startswith('.'): continue
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<50s}  {size_kb:>8.1f} KB")